# 05 · Training Pipeline, Quality Gate and Model Registry

```
PrepareData -> TrainBaseline -> TrainCandidate -> Evaluate -> QualityGate --pass--> RegisterTagGate (Model Registry)
                                                                          \--fail--> QualityGateFailed (FailStep)
```

* **PrepareData** validates the Feature Store snapshot (schema, row count, positives per tag) and writes the splits plus
  the Model Monitor baseline dataset.
* **QualityGate** registers the candidate only if test macro-F1 >= `MacroF1Threshold` **and** it beats the baseline.
* We run the pipeline twice: with defaults (**success**) and with `MacroF1Threshold=0.95` (**intentional failure**).

Open *SageMaker Studio > Pipelines > steam-tag-gate-pipeline* to see both DAGs.

In [ ]:
# Pin the SageMaker SDK v2 API used by this project (no-op if already installed)
%pip install -q "sagemaker>=2.232,<3"

In [ ]:
import os
import sys
import warnings

warnings.filterwarnings("ignore")
def find_repo_root():
    """The notebooks import from src/, so the whole repo must be cloned, not just notebooks/."""
    candidates = [os.path.expanduser("~/steam-store-copilot")]
    try:  # the kernel's working directory may have been deleted (e.g. after re-cloning)
        candidates = [os.getcwd(), os.path.dirname(os.getcwd())] + candidates
    except FileNotFoundError:
        os.chdir(os.path.expanduser("~"))
    for path in candidates:
        if os.path.isdir(os.path.join(path, "src")) and os.path.exists(os.path.join(path, "src", "config.py")):
            return path
    raise RuntimeError("Could not find the repo's src/ folder. Clone the full repository in Studio "
                       "(Git > Clone Repository) and open the notebooks from steam-store-copilot/notebooks/.")


REPO_ROOT = find_repo_root()
if REPO_ROOT not in sys.path:
    sys.path.insert(0, REPO_ROOT)
print("Repo root:", REPO_ROOT)

import boto3
import sagemaker

from src.config import S3Layout

sess = sagemaker.Session()
region = sess.boto_region_name
role = sagemaker.get_execution_role()  # LabRole in AWS Academy Learner Lab
layout = S3Layout(bucket=sess.default_bucket())
sm = boto3.client("sagemaker", region_name=region)
print("Region:", region)
print("Role:  ", role)
print("Bucket:", layout.bucket)

In [ ]:
import json

import pandas as pd

from src.config import TAG_GATE_MODEL_GROUP
from src.data.s3io import read_text
from src.pipelines.tag_gate_pipeline import get_pipeline

pipeline = get_pipeline(role=role, bucket=layout.bucket, region=region)
pipeline.upsert(role_arn=role, description="Tag QA Gate training pipeline")
definition = json.loads(pipeline.definition())
pd.DataFrame([{"step": s["Name"], "type": s["Type"]} for s in definition["Steps"]])

## Run 1: default parameters (expected: **Succeeded**)

In [ ]:
def run_and_report(execution):
    try:
        execution.wait(delay=30, max_attempts=180)
    except Exception as exc:  # the waiter raises when the execution fails
        print("Waiter:", type(exc).__name__)
    status = execution.describe()["PipelineExecutionStatus"]
    steps = pd.DataFrame([{"step": s["StepName"], "status": s["StepStatus"],
                           "reason": s.get("FailureReason", "")[:120]} for s in execution.list_steps()])
    print("Pipeline status:", status)
    return status, steps[::-1].reset_index(drop=True)


ok_execution = pipeline.start(execution_display_name="default-run")
ok_status, ok_steps = run_and_report(ok_execution)
ok_steps

In [ ]:
execution_id = ok_execution.arn.split("/")[-1]
evaluation = json.loads(read_text(f"{layout.pipeline_artifacts}/{execution_id}/evaluation/evaluation.json"))
pd.DataFrame({k: v["value"] for k, v in evaluation["tag_gate"].items()}, index=["candidate (test)"]).T.round(4)

## Run 2: unreachable quality bar (expected: **Failed** at the QualityGate)
This demonstrates the failed state of the DAG: the candidate is evaluated but **not** registered.

In [ ]:
fail_execution = pipeline.start(parameters={"MacroF1Threshold": 0.95}, execution_display_name="quality-gate-fail-demo")
fail_status, fail_steps = run_and_report(fail_execution)
fail_steps

In [ ]:
assert ok_status == "Succeeded", ok_status
assert fail_status == "Failed", fail_status
print("Both CI/CD states demonstrated: success -> registered, failure -> blocked at the quality gate.")

## Model Registry
New versions arrive as `PendingManualApproval`. Approving the latest version is the human sign-off that promotes it to
batch inference (notebook 06).

In [ ]:
packages = sm.list_model_packages(ModelPackageGroupName=TAG_GATE_MODEL_GROUP, SortBy="CreationTime",
                                  SortOrder="Descending")["ModelPackageSummaryList"]
pd.DataFrame(packages)[["ModelPackageVersion", "ModelPackageStatus", "ModelApprovalStatus", "CreationTime"]]

In [ ]:
latest = packages[0]["ModelPackageArn"]
sm.update_model_package(ModelPackageArn=latest, ModelApprovalStatus="Approved",
                        ApprovalDescription="Meets macro-F1 and false-approval targets")
detail = sm.describe_model_package(ModelPackageName=latest)
print("Approved:", latest)
print("Metrics: ", detail["ModelMetrics"]["ModelQuality"]["Statistics"]["S3Uri"]
      if "ModelQuality" in detail.get("ModelMetrics", {}) else detail["ModelMetrics"])
print("Image:   ", detail["InferenceSpecification"]["Containers"][0]["Image"])

## CI side (GitHub Actions)
`.github/workflows/ci.yml` runs on every push and pull request: **lint -> unit tests -> pipeline definition build**,
and on `main`, a **deploy** job that runs `scripts/run_pipeline.py` (this same pipeline) when AWS credentials are
configured as repository secrets. The Actions tab shows a green and a red run (the red run is a PR that reintroduced
the CSV-header bug and was caught by `tests/test_loaders.py`).